In [1]:
import numpy as np
import pandas as pd

from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print(Path.cwd())

from core.forecasting.dataset import load_daily_sales, create_daily_series

from core.forecasting.baseline import create_baseline_features

from core.forecasting.evaluation import evaluate_forecast

from core.forecasting.intermittent import croston_forecast, sba_forecast, evaluate_intermittent_forecast

daily_sales = load_daily_sales()
forecasting_data = create_daily_series(daily_sales)

E:\Programming\Python\StockWise


In [2]:
# 1. Evaluate Baseline

forecasting_data = create_baseline_features(
    forecasting_data
)

# Only rows where the 7-day baseline exists.
evaluation_data = forecasting_data.dropna(
    subset=["BaselineForecast"]
).copy()

# Time-based holdout.
cutoff_date = pd.Timestamp("2011-09-26")

train = evaluation_data[
    evaluation_data["Date"] < cutoff_date
]

test = evaluation_data[
    evaluation_data["Date"] >= cutoff_date
]

metrics = evaluate_forecast(
    test["Demand"],
    test["BaselineForecast"],
)

print("--- Baseline Evaluation ---")

print(
    f"Training period: "
    f"{train['Date'].min().date()} "
    f"to "
    f"{train['Date'].max().date()}"
)

print(
    f"Test period: "
    f"{test['Date'].min().date()} "
    f"to "
    f"{test['Date'].max().date()}"
)

print(
    f"Training rows: "
    f"{len(train):,}"
)

print(
    f"Test rows: "
    f"{len(test):,}"
)

print(
    f"\nMAE:  {metrics['MAE']:.2f}"
)

print(
    f"RMSE: {metrics['RMSE']:.2f}"
)

--- Baseline Evaluation ---
Training period: 2010-12-08 to 2011-09-25
Test period: 2011-09-26 to 2011-12-09
Training rows: 840,877
Test rows: 210,460

MAE:  10.12
RMSE: 62.19


In [3]:
# 2. Croston / SBA / intermittent forecasting

examples = {
    "regular": np.array(
        [10, 10, 10, 10, 10]
    ),
    "intermittent": np.array(
        [0, 0, 10, 0, 0, 0, 20, 0, 0]
    ),
    "lumpy": np.array(
        [0, 0, 50, 0, 0, 5, 0, 0, 100]
    ),
    "zero": np.array(
        [0, 0, 0, 0]
    ),
}

for name, demand in examples.items():

    croston_value = croston_forecast(demand)
    sba_value = sba_forecast(demand)

    print(
        name,
        "Croston:",
        round(croston_value, 4),
        "SBA:",
        round(sba_value, 4),
    )

regular Croston: 10.0 SBA: 9.5
intermittent Croston: 2.4719 SBA: 2.3483
lumpy Croston: 16.9833 SBA: 16.1342
zero Croston: 0.0 SBA: 0.0


In [4]:
croston_results = evaluate_intermittent_forecast(
    forecasting_data,
    cutoff="2011-09-26",
    method="croston",
)

sba_results = evaluate_intermittent_forecast(
    forecasting_data,
    cutoff="2011-09-26",
    method="sba",
)

croston_metrics = evaluate_forecast(
    croston_results["Demand"],
    croston_results["Forecast"],
)

sba_metrics = evaluate_forecast(
    sba_results["Demand"],
    sba_results["Forecast"],
)

print("--- Intermittent Forecast Evaluation ---")

print(
    f"Croston MAE:  "
    f"{croston_metrics['MAE']:.2f}"
)

print(
    f"Croston RMSE: "
    f"{croston_metrics['RMSE']:.2f}"
)

print()

print(
    f"SBA MAE:      "
    f"{sba_metrics['MAE']:.2f}"
)

print(
    f"SBA RMSE:     "
    f"{sba_metrics['RMSE']:.2f}"
)

--- Intermittent Forecast Evaluation ---
Croston MAE:  9.03
Croston RMSE: 46.98

SBA MAE:      8.90
SBA RMSE:     46.98


In [5]:
comparison = pd.DataFrame(
    {
        "Model": [
            "Seasonal Naive",
            "Croston",
            "SBA",
        ],
        "MAE": [
            metrics["MAE"],
            croston_metrics["MAE"],
            sba_metrics["MAE"],
        ],
        "RMSE": [
            metrics["RMSE"],
            croston_metrics["RMSE"],
            sba_metrics["RMSE"],
        ],
    }
)

print("--- Forecast Model Comparison ---")

print(
    comparison
    .sort_values("MAE")
    .round(2)
    .to_string(index=False)
)

--- Forecast Model Comparison ---
         Model   MAE  RMSE
           SBA  8.90 46.98
       Croston  9.03 46.98
Seasonal Naive 10.12 62.19
